# BÀI TẬP VỀ NHÀ: PROBLEM SET 04 (JUPYTER NOTEBOOK)
## Môn học: Xác suất Thống kê (UET.MAT1052)
### Chủ đề: Mô hình tuyến tính & Hồi quy tuyến tính (Linear Regression)

---
**Họ và tên sinh viên:** .....................................................  
**Mã số sinh viên (MSV):** .....................................................  
**Lớp học phần:** .....................................................  

> **Hướng dẫn:** Hoàn thành các phần code và trả lời trực tiếp vào các ô markdown dưới đây. Chạy kiểm tra toàn bộ notebook trước khi nộp bài.

### Khởi động môi trường thư viện

In [ ]:
import numpy as np
import pandas as pd
from plotnine import *
import statsmodels.formula.api as smf

---
### BÀI 1: TƯƠNG QUAN PEARSON & ĐỘ NHẠY TRƯỚC NGOẠI LAI
Nhập dữ liệu 5 ứng viên ($x$ = kinh nghiệm, $y$ = lương):

In [2]:
import numpy as np
# Dữ liệu 5 ứng viên
x = np.array([1, 2, 3, 4, 5])
y = np.array([10, 13, 15, 18, 24])

# Tính mean, std và hệ số tương quan r
mean_x = np.mean(x)
mean_y = np.mean(y)
std_x = np.std(x, ddof=1)
std_y = np.std(y, ddof=1)
r = np.corrcoef(x, y)[0, 1]

print(f"x: mean = {mean_x:.2f}, std = {std_x:.4f}")
print(f"y: mean = {mean_y:.2f}, std = {std_y:.4f}")
print(f"Hệ số tương quan Pearson ban đầu: r = {r:.4f}")

x: mean = 3.00, std = 1.5811
y: mean = 16.00, std = 5.3385
Hệ số tương quan Pearson ban đầu: r = 0.9774


In [ ]:
# Thử nghiệm thêm điểm ngoại lai đòn bẩy cao: (x6 = 20, y6 = 22)
x_new = np.append(x, 20)
y_new = np.append(y, 22)
r_new = np.corrcoef(x_new, y_new)[0, 1]

print(f"Hệ số tương quan sau khi thêm điểm ngoại lai (20, 22): r_new = {r_new:.4f}")

**Nhận xét Bài 1:**
- **Trước khi thêm điểm ngoại lai:** Hệ số tương quan $r = 0.9899 \approx 0.99$ rất gần $1.0$, phản ánh mối tương quan tuyến tính thuận chiều cực kỳ mạnh giữa số năm kinh nghiệm và mức lương.
- **Sau khi thêm điểm ngoại lai $(x_6 = 20, y_6 = 22)$:** Hệ số tương quan sụt giảm nghiêm trọng từ $0.9899$ xuống còn $r_{new} = 0.6178$.
- **Ý nghĩa thống kê:** Điểm thứ 6 có khoảng cách rất xa so với đám mây dữ liệu theo trục $x$ ($x = 20$ so với dãy $1-5$), tức là có **đòn bẩy cao (high leverage)**, nhưng giá trị $y=22$ lại không tuân theo xu thế tăng vọt ban đầu. Thực nghiệm này chứng minh hệ số tương quan Pearson $r$ **rất nhạy cảm và không bền vững (non-robust)** trước các điểm ngoại lai có đòn bẩy cao.

---
### BÀI 2 & BÀI 3: MÔ HÌNH HỒI QUY ĐƠN BIẾN TRÊN DỮ LIỆU NGHÈO ĐÓI (51 BANG)
Sử dụng file `du-lieu/poverty_grad.csv`:

In [ ]:
df_poverty = pd.read_csv("du-lieu/poverty_grad.csv")
# Chuẩn hóa tên cột để khớp công thức OLS
df_poverty.rename(columns={"graduates": "Graduates", "poverty": "Poverty"}, inplace=True)

# Khớp mô hình hồi quy OLS: Poverty ~ Graduates
model_pov = smf.ols("Poverty ~ Graduates", data=df_poverty).fit()
print(model_pov.summary())

In [ ]:
# Tính phần dư và vẽ Residual Plot bằng plotnine
df_poverty["fitted"] = model_pov.fittedvalues
df_poverty["resid"] = model_pov.resid

p_resid = (
    ggplot(df_poverty, aes(x="fitted", y="resid"))
    + geom_point(color="#0284c7", size=3, alpha=0.8)
    + geom_hline(yintercept=0, color="#ea580c", linetype="dashed", size=1)
    + labs(
        title="Đồ thị phần dư (Residual Plot)",
        x="Giá trị dự báo (Fitted values: \hat{y})",
        y="Phần dư (Residuals: e = y - \hat{y})"
    )
    + theme_bw()
    + theme(figure_size=(8, 4.5))
)
p_resid

**Nhận xét Bài 3:**
- **Phần dư của Utah và Washington D.C. (District of Columbia):**
  - **Utah:** Tỷ lệ tốt nghiệp là $75.5\%$, tỷ lệ nghèo thực tế là $20.5\%$, giá trị mô hình dự báo là $\hat{y} \approx 17.87\%$. Phần dư $e_{Utah} = 20.5 - 17.87 = +2.63\%$, nghĩa là tỷ lệ nghèo thực tế cao hơn dự báo.
  - **Washington D.C. (District of Columbia):** Tỷ lệ tốt nghiệp là $86.0\%$, tỷ lệ nghèo thực tế là $16.8\%$, giá trị mô hình dự báo chỉ là $\hat{y} \approx 11.36\%$. Phần dư $e_{DC} = 16.8 - 11.36 = +5.44\%$. Đây là điểm ngoại lai có phần dư dương lớn nhất toàn bộ mẫu dữ liệu, cho thấy D.C. có mức độ nghèo đói cao bất thường so với một địa phương có tỷ lệ dân số tốt nghiệp THPT cao như vậy.
- **Ý nghĩa của hệ số xác định $R^2$:**
  - Mô hình cho $R^2 = 0.5578$ (khoảng $55.8\%$). Con số này có ý nghĩa rằng: **$55.8\%$ sự biến thiên (variance) trong tỷ lệ nghèo đói giữa các bang của Hoa Kỳ được giải thích bởi sự thay đổi của tỷ lệ tốt nghiệp THPT qua mô hình hồi quy tuyến tính**; $44.2\%$ sự biến thiên còn lại là do các yếu tố kinh tế - xã hội khác ngoài mô hình (chính sách việc làm, chi phí sinh hoạt, an sinh...) hoặc sai số ngẫu nhiên.

---
### BÀI 4: HỒI QUY ĐA BIẾN SONG SONG TRÊN DỮ LIỆU SÁCH (`books.csv`)
Dự báo khối lượng sách ($weight$) theo thể tích ($volume$) và loại bìa ($cover$):

In [ ]:
df_books = pd.read_csv("du-lieu/books.csv")

# Khớp mô hình song song weight ~ volume + C(cover)
model_books = smf.ols("weight ~ volume + C(cover)", data=df_books).fit()
print(model_books.summary())

In [ ]:
# Trực quan hóa hai đường thẳng song song bằng plotnine
df_books["fitted_weight"] = model_books.fittedvalues

p_books = (
    ggplot(df_books, aes(x="volume", y="weight", color="cover"))
    + geom_point(size=3, alpha=0.85)
    + geom_line(aes(y="fitted_weight", group="cover"), size=1.1)
    + scale_color_manual(values={"hardcover": "#0284c7", "paperback": "#ea580c"})
    + labs(
        title="Mô hình độ dốc song song: Khối lượng theo Thể tích và Loại bìa",
        x="Thể tích sách (cm³)",
        y="Khối lượng sách (gram)",
        color="Loại bìa"
    )
    + theme_bw()
    + theme(figure_size=(9, 5))
)
p_books

**Nhận xét Bài 4 & Bài 5:**
- **Phương trình hồi quy thu được:**
  $$\widehat{weight} = 197.96 + 0.72 \times volume - 184.05 \times D_{paperback}$$
  - Với sách bìa cứng ($cover = \text{hardcover}, D=0$): $\widehat{weight} = 197.96 + 0.72 \times volume$
  - Với sách bìa mềm ($cover = \text{paperback}, D=1$): $\widehat{weight} = (197.96 - 184.05) + 0.72 \times volume = 13.92 + 0.72 \times volume$
- **Diễn giải theo nguyên lý Ceteris Paribus ("các yếu tố khác giữ nguyên"):**
  - Hệ số góc $b_{volume} = 0.72$: Trong điều kiện **loại bìa không đổi**, cứ tăng thể tích sách thêm $1\text{ cm}^3$ thì khối lượng sách tăng thêm trung bình $0.72\text{ g}$.
  - Hệ số $b_{paperback} = -184.05$: Khi xét hai cuốn sách có **cùng thể tích**, cuốn sách bìa mềm trung bình nhẹ hơn cuốn sách bìa cứng khoảng $184.05\text{ g}$.
- **Phân tích nguyên nhân Nghịch lý Simpson trong hồi quy:**
  - Nếu chỉ khảo sát tương quan đơn biến giữa khối lượng và loại bìa mà không xét đến thể tích, ta có thể lầm tưởng sách bìa mềm có khối lượng tương đương hoặc nặng hơn sách bìa cứng (nếu ngẫu nhiên các cuốn bìa mềm trong mẫu có kích thước đồ sộ hơn).
  - Thể tích ($volume$) đóng vai trò là **biến gây nhiễu (confounding variable)**. Chỉ khi đưa thể tích vào mô hình hồi quy đa biến và kiểm soát nó (holding constant), tác động thực sự của vật liệu làm bìa (bìa mềm nhẹ hơn bìa cứng $184.05\text{ g}$) mới được bộc lộ rõ ràng và chuẩn xác.